# SyftBox Manual E2E Test

This notebook walks through a complete end-to-end workflow between a **Data Owner (DO)** and a **Data Scientist (DS)** using SyftBox. It demonstrates dataset creation, browsing, job submission, code review, approval/rejection, execution, and result retrieval.

### Flow Overview

1. **Setup** — Configuration, authentication, and peer connection
2. **Data Owner** — Creates and publishes datasets
3. **Data Scientist** — Browses datasets, prototypes code on mock data, submits two jobs
4. **Data Owner** — Reviews submitted code, approves one job, rejects the other, and runs the approved job
5. **Data Scientist** — Retrieves the output of the approved job and the rejection reason for the other

### Prerequisites
- A `.env` file with: `DO_EMAIL`, `DS_EMAIL`, `TOKEN_DO`, `TOKEN_DS`

---

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

### 1.1 Configuration

Load environment variables and validate that credential files exist.

In [2]:
import os
from pathlib import Path

# Load .env file
for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]

do_token_path = Path(os.environ["TOKEN_DO"])
ds_token_path = Path(os.environ["TOKEN_DS"])

assert do_token_path.exists() and ds_token_path.exists()

### 1.3 Login

Log in as both the Data Owner and the Data Scientist. Each client maintains its own local SyftBox folder and syncs independently.

In [3]:
import syft_client as sc
do_client = sc.login_do(email=DO_EMAIL, token_path=do_token_path)

Created user directory: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org
Ensured directories exist: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org/app_data/job
🔑 Logging in as sameer@openmined.org...
   Environment: Python
No checkpoints found, downloading all events...

✅ Logged in successfully!
   SyftBox folder : /Users/swag/SyftBox_sameer@openmined.org
   Version        : 0.1.112

💡 No peers yet. Add a Data Owner with:
   client.add_peer('do@org.com')


In [4]:
ds_client = sc.login_ds(email=DS_EMAIL, token_path=ds_token_path)

Created user directory: /Users/swag/SyftBox_som.wom.beach@gmail.com/som.wom.beach@gmail.com
🔑 Logging in as som.wom.beach@gmail.com...
   Environment: Python

✅ Logged in successfully!
   SyftBox folder : /Users/swag/SyftBox_som.wom.beach@gmail.com
   Version        : 0.1.112

💡 No peers yet. Add a Data Owner with:
   client.add_peer('do@org.com')


### 1.4 Optional — Wipe prior state

Uncomment to delete all local SyftBox data and start fresh.

In [5]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()

### 1.5 Establish peer connection

The Data Scientist requests access to the Data Owner, and the Data Owner approves. This is required before any data or jobs can be exchanged.

In [6]:
ds_client.add_peer(DO_EMAIL)
do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)

print("Peer relationship established")

🤝 Sending peer request to sameer@openmined.org...

✅ Peer request sent to sameer@openmined.org!
   ⏳ Next step: ask them to approve your request.
   Once approved, run client.sync() to confirm the connection.
Approved peer request from som.wom.beach@gmail.com
Peer relationship established


In [7]:
do_client.peers

PeerList([Peer(email='som.wom.beach@gmail.com', platforms=[GdriveFilesPlatform(name='Gdrive Files', module_path='google_personal.gdrive_files')], state=<PeerState.ACCEPTED: 'accepted'>, version=VersionInfo(syft_client_version='0.1.112', min_supported_syft_client_version='0.1.93', protocol_version='1.0.0', min_supported_protocol_version='1.0.0', updated_at=datetime.datetime(2026, 5, 4, 20, 9, 51, 480834, tzinfo=TzInfo(0))), public_encryption_bundle=None, use_encryption=False)])

---

## 2. Data Owner: Create Datasets

The Data Owner creates datasets that will be shared with the Data Scientist. Each dataset has:
- **Mock data** — a public, non-sensitive version that the DS can freely access and prototype against
- **Private data** — the real data that never leaves the DO's machine; only job code can access it at runtime

### 2.1 Create a simple text dataset

In [8]:
do_client.create_dataset(
    name="testdata",
    mock_path="data/mock.txt",
    private_path="data/private.txt",
    users=[DS_EMAIL],
)

name,testdata
created_at,2026-05-04 20:10:25
summary,None
tags,[]
location,None


### 2.2 Create a PDF dataset

Generate sample PDFs, then create a dataset with a CSV manifest as mock data and the PDFs as private data.

In [9]:
!uv run create_test_pdfs.py

Created 11 PDFs in /Users/swag/Documents/Coding/syft-client/notebooks/beach/internal/data/PDFs
Created /Users/swag/Documents/Coding/syft-client/notebooks/beach/internal/data/manifest.csv
Created /Users/swag/Documents/Coding/syft-client/notebooks/beach/internal/data/readme.md


In [10]:
do_client.create_dataset(
    name="pdfdata",
    mock_path="data/manifest.csv",
    private_path="data/PDFs",
    summary="A dataset with some PDFs",
    readme_path="data/readme.md",
    tags=["test", "pdf"],
    users=[DS_EMAIL],
    upload_private=False,
    sync=True,
)

name,pdfdata
created_at,2026-05-04 20:10:43
summary,A dataset with some PDFs
tags,"['test', 'pdf']"
location,None


### 2.3 View datasets

The Data Owner can list all their datasets and inspect individual ones.

In [11]:
do_client.datasets

SyftDatasetManager(2 datasets)

In [12]:
dataset = do_client.datasets.get("testdata", datasite=DO_EMAIL)
dataset

name,testdata
created_at,2026-05-04 20:10:25
summary,None
tags,[]
location,None


In [13]:
mock_contents = dataset.mock_files[0].read_text()
print(f"Mock file contents: {mock_contents!r}")

Mock file contents: 'mock'


---

## 3. Data Scientist: Browse Datasets and Prototype Code

The Data Scientist syncs to discover available datasets. They can only see mock data — the private data stays on the Data Owner's machine.

### 3.1 List available datasets

In [14]:
ds_client.datasets

SyftDatasetManager(2 datasets)

### 3.2 Write trial code against mock data

Before submitting a job, the Data Scientist can prototype their analysis locally using the mock data. The `resolve_dataset_file_path` function returns mock data when run locally, and private data when run as a job on the DO's machine.

> **Note:** Remove the `client=` parameter before submitting as a job — it will resolve automatically on the DO's machine.

In [15]:
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata", client=ds_client)
df = pd.read_fwf(resolved_path)
print(f"Type of data in use: {df.columns[0]}")

📋 Resolving MOCK files for 'testdata' (local test mode)...

💡 Remove client=client before submitting — the real job, it will automatically be resolved on the DO machine
Type of data in use: mock


### 3.3 Submit jobs

Now the Data Scientist submits two different jobs to the Data Owner. Each job is a self-contained Python script that will run in an isolated environment on the DO's machine.

- **Job A** (`summary_stats`) — computes basic statistics on the dataset
- **Job B** (`column_names`) — extracts and returns column names from the dataset

Both jobs use `resolve_dataset_file_path` to access the private data at runtime.

In [16]:
import tempfile

def write_job_script(code: str) -> str:
    """Write a job script to a temp file and return its path."""
    job_dir = Path(tempfile.mkdtemp(prefix="syft_job_"))
    script = job_dir / "main.py"
    script.write_text(code)
    return str(script)

#### Job A: Summary Statistics

A simple analysis job that reads the dataset and computes summary statistics.

In [17]:
JOB_A_NAME = "summary_stats"

job_a_code = write_job_script('''import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

result = {
    "message": "Summary statistics computed successfully",
    "row_count": len(df),
    "columns": list(df.columns),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_a_code,
    job_name=JOB_A_NAME,
    force_submission=True,
    dependencies=["pandas"],
)
print(f"Job '{JOB_A_NAME}' submitted to {DO_EMAIL}")

📤 Submitting '/var/folders/6j/cbsd565s2rvfght9w8pgv_t80000gn/T/syft_job_dktbqa80/main.py' to sameer@openmined.org...
   Job name     : summary_stats
   Dependencies : pandas

✅ Job submitted successfully!
   Status : inbox (waiting for DO to review)

⏳ Next step: wait for sameer@openmined.org to approve and run it.
   Check progress with: client.jobs
Job 'summary_stats' submitted to sameer@openmined.org


#### Job B: Column Names

This job intentionally leaks column names from the private dataset — the kind of thing a Data Owner might want to reject.

In [18]:
JOB_B_NAME = "column_names"

job_b_code = write_job_script('''import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

# This leaks sensitive column names from the private dataset
result = {
    "message": "Extracted column names",
    "column_names": list(df.columns),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_b_code,
    job_name=JOB_B_NAME,
    force_submission=True,
    dependencies=["pandas"],
)
print(f"Job '{JOB_B_NAME}' submitted to {DO_EMAIL}")

📤 Submitting '/var/folders/6j/cbsd565s2rvfght9w8pgv_t80000gn/T/syft_job_lxlhpido/main.py' to sameer@openmined.org...
   Job name     : column_names
   Dependencies : pandas

✅ Job submitted successfully!
   Status : inbox (waiting for DO to review)

⏳ Next step: wait for sameer@openmined.org to approve and run it.
   Check progress with: client.jobs
Job 'column_names' submitted to sameer@openmined.org


#### Job C: Timeout Job
A simple job that runs an infinite loop, intended to test the timeout functionality.

In [19]:
JOB_C_NAME = "infinite_loop"

job_c_code = write_job_script('''                              
import time
while True:
    time.sleep(1)
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_c_code,
    job_name=JOB_C_NAME,
    force_submission=True,
)

print(f"Job '{JOB_C_NAME}' submitted to {DO_EMAIL}")

📤 Submitting '/var/folders/6j/cbsd565s2rvfght9w8pgv_t80000gn/T/syft_job_9mavuqjh/main.py' to sameer@openmined.org...
   Job name     : infinite_loop

✅ Job submitted successfully!
   Status : inbox (waiting for DO to review)

⏳ Next step: wait for sameer@openmined.org to approve and run it.
   Check progress with: client.jobs
Job 'infinite_loop' submitted to sameer@openmined.org


#### Job D: Pre-Submit Scanner Validation

This submission contains `client=ds_client` in the resolver call, which would fail on the DO's machine. The pre-submit scanner should detect this and prompt the user to confirm. Typing "N" (or just pressing Enter) blocks the submission.

In [20]:
# Job D: code contains client=ds_client — pre-submit scanner should catch this
# When prompted, type "N" to block the submission

job_d_code = write_job_script('''import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata", client=ds_client)
df = pd.read_fwf(resolved_path)
print(df.describe())
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_d_code,
    job_name="blocked_client_ds",
)

🚨 Pre-submit check failed:

   Your script contains 'client=client' which will fail on the DO's machine.
   'client' is not defined there — it is only available in your local Colab session.

   Fix: remove 'client=client' from resolve_dataset_files_path() and re-run %%writefile.

   ✅ Correct:  sc.resolve_dataset_files_path('beach_water_quality')
   ❌ Wrong:    sc.resolve_dataset_files_path('beach_water_quality', client=client)



Continue with submission anyway? (y/N):  N


Submission aborted.


#### Job E: PDF Size Computation

A job that uses `resolve_dataset_files_path` to get all PDF file paths from the `pdfdata` dataset and sums their sizes in bytes.

In [21]:
JOB_E_NAME = "pdf_size_sum"

job_e_code = write_job_script('''import os, json
import syft_client as sc

dataset_files = sc.resolve_dataset_files_path("pdfdata")

total_bytes = 0
file_count = 0
for file_path in dataset_files:
    size = os.path.getsize(file_path)
    total_bytes += size
    file_count += 1

result = {
    "message": "PDF size computation completed successfully",
    "file_count": file_count,
    "total_bytes": total_bytes,
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_e_code,
    job_name=JOB_E_NAME,
    force_submission=True,
)
print(f"Job '{JOB_E_NAME}' submitted to {DO_EMAIL}")

📤 Submitting '/var/folders/6j/cbsd565s2rvfght9w8pgv_t80000gn/T/syft_job_8uxrdyi3/main.py' to sameer@openmined.org...
   Job name     : pdf_size_sum

✅ Job submitted successfully!
   Status : inbox (waiting for DO to review)

⏳ Next step: wait for sameer@openmined.org to approve and run it.
   Check progress with: client.jobs
Job 'pdf_size_sum' submitted to sameer@openmined.org


#### Job F: Missing Dependency (Execution Failure)

Same code as Job A but submitted without `dependencies=["pandas"]`. This job will be approved but fail during execution because pandas is not installed in the isolated venv.

In [22]:
JOB_F_NAME = "missing_dependency"

job_f_code = write_job_script('''import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

result = {
    "message": "Summary statistics computed successfully",
    "row_count": len(df),
    "columns": list(df.columns),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

# Note: no dependencies=["pandas"] — this will cause execution failure
ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_f_code,
    job_name=JOB_F_NAME,
    force_submission=True,
)
print(f"Job '{JOB_F_NAME}' submitted to {DO_EMAIL}")

📤 Submitting '/var/folders/6j/cbsd565s2rvfght9w8pgv_t80000gn/T/syft_job_lpb_7rvz/main.py' to sameer@openmined.org...
   Job name     : missing_dependency

✅ Job submitted successfully!
   Status : inbox (waiting for DO to review)

⏳ Next step: wait for sameer@openmined.org to approve and run it.
   Check progress with: client.jobs
Job 'missing_dependency' submitted to sameer@openmined.org


---

## 4. Data Owner: Review, Approve/Reject, and Run Jobs

The Data Owner syncs to receive the submitted jobs, reviews the code, and decides whether to approve or reject each one. Approved jobs are then executed on the DO's machine.

### 4.1 List incoming jobs

In [23]:
do_client.jobs

Uploading rolling state with 5 events...
Uploading rolling state with 8 events...
Uploading rolling state with 11 events...
Uploading rolling state with 14 events...
Uploading rolling state with 17 events...


Index,Job Name,Submitted By,Status,Approval
[0],missing_dependency,som.wom.beach@gmail.com,📥 PENDING,—
[1],pdf_size_sum,som.wom.beach@gmail.com,📥 PENDING,—
[2],infinite_loop,som.wom.beach@gmail.com,📥 PENDING,—
[3],column_names,som.wom.beach@gmail.com,📥 PENDING,—
[4],summary_stats,som.wom.beach@gmail.com,📥 PENDING,—


### 4.2 Review the submitted code

Look up each job by name and inspect the code before making a decision.

In [24]:
job_a = do_client.jobs[JOB_A_NAME]
job_b = do_client.jobs[JOB_B_NAME]
job_c = do_client.jobs[JOB_C_NAME]
job_e = do_client.jobs[JOB_E_NAME]
job_f = do_client.jobs[JOB_F_NAME]
print(f"--- {job_a.name} (status: {job_a.status}) ---")
print(job_a.code)

--- summary_stats (status: pending) ---
import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

result = {
    "message": "Summary statistics computed successfully",
    "row_count": len(df),
    "columns": list(df.columns),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")



### 4.3 View the full job object
Alternatively, you can view the full job object.

In [25]:
job_a

JobInfo(name='summary_stats', submitted_by='som.wom.beach@gmail.com', current_user_email='sameer@openmined.org', status='pending')

### 4.4 Approve/Reject Jobs

The DO approves Jobs A, C, E, and F, and rejects Job B. Job F is intentionally approved so it can demonstrate execution failure.

In [26]:
job_a.approve()

✅ Job 'summary_stats' approved successfully!
   Status    : approved → will run on next process cycle

⏳ Next step: run process_approved_jobs() to execute it.
   client.process_approved_jobs(share_outputs_with_submitter=True)


In [27]:
job_b.reject(reason="The job leaks sensitive column names from the private dataset")

Job 'column_names' rejected.


In [28]:
job_c.approve()

✅ Job 'infinite_loop' approved successfully!
   Status    : approved → will run on next process cycle

⏳ Next step: run process_approved_jobs() to execute it.
   client.process_approved_jobs(share_outputs_with_submitter=True)


In [29]:
job_e.approve()

✅ Job 'pdf_size_sum' approved successfully!
   Status    : approved → will run on next process cycle

⏳ Next step: run process_approved_jobs() to execute it.
   client.process_approved_jobs(share_outputs_with_submitter=True)


In [30]:
job_f.approve()

✅ Job 'missing_dependency' approved successfully!
   Status    : approved → will run on next process cycle

⏳ Next step: run process_approved_jobs() to execute it.
   client.process_approved_jobs(share_outputs_with_submitter=True)


### 4.5 Run approved jobs

Execute all approved jobs. Each job runs in an isolated Python virtual environment on the DO's machine. The private dataset is accessible to the job code via `resolve_dataset_file_path`. You can pass these arguments: 
1. `stream_output`: If True (default), stream output in real-time. If False, capture output at end.
2. `timeout`: Timeout in seconds per job. Defaults to 300 (5 minutes).
3. `force_execution`: If True, process all approved jobs regardless of version compatibility. If False (default), skip jobs from peers with incompatible or unknown versions.
4. `share_outputs_with_submitter`: If True, grant read access on outputs to submitter.
5. `share_logs_with_submitter`: If True, grant read access on logs to submitter.

In [31]:
do_client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,
)

 Found 4 job(s) in approved status

 Executing job: summary_stats
 Inbox: /Users/swag/SyftBox_sameer@openmined.org/sameer@openmined.org/app_data/job/inbox/som.wom.beach@gmail.com/summary_stats
[sameer@openmined.org][summary_stats] STDERR: Using CPython 3.12.10 interpreter at: /opt/homebrew/opt/python@3.12/bin/python3.12
[sameer@openmined.org][summary_stats] STDERR: Creating virtual environment at: .venv
[sameer@openmined.org][summary_stats] STDERR: Activate with: source .venv/bin/activate
[sameer@openmined.org][summary_stats] STDERR: Resolved 90 packages in 358ms
[sameer@openmined.org][summary_stats] STDERR: Installed 90 packages in 669ms
[sameer@openmined.org][summary_stats] STDERR:  + annotated-types==0.7.0
[sameer@openmined.org][summary_stats] STDERR:  + asttokens==3.0.1
[sameer@openmined.org][summary_stats] STDERR:  + bracex==2.6
[sameer@openmined.org][summary_stats] STDERR:  + certifi==2026.4.22
[sameer@openmined.org][summary_stats] STDERR:  + cffi==2.0.0
[sameer@openmined.org][su

---

## 5. Data Scientist: Retrieve Results

The Data Scientist syncs to pull back the job results. They can read the output of the approved job and see the reason for the rejected one.

### 5.1 Check job statuses

In [32]:
ds_client.jobs

Index,Job Name,Submitted By,Status,Approval
[0],missing_dependency,som.wom.beach@gmail.com,📨 RECEIVED,—
[1],pdf_size_sum,som.wom.beach@gmail.com,📨 RECEIVED,—
[2],infinite_loop,som.wom.beach@gmail.com,📨 RECEIVED,—
[3],column_names,som.wom.beach@gmail.com,📨 RECEIVED,—
[4],summary_stats,som.wom.beach@gmail.com,📨 RECEIVED,—


### 5.2 Read the output of the approved job

In [ ]:
ds_job_a = ds_client.jobs[JOB_A_NAME]
print(f"Status: {ds_job_a.status}")
print(f"Output files: {ds_job_a.output_paths}")

if ds_job_a.output_paths:
    print(f"\nResult:\n{ds_job_a.output_paths[0].read_text()}")

### 5.3 See the rejection reason for the rejected job

In [ ]:
ds_job_b = ds_client.jobs[JOB_B_NAME]
print(f"Status: {ds_job_b.status}")
print(f"Rejection reason: {ds_job_b.review_reason}")

### 5.4 View the timeout job status

In [ ]:
ds_job_c = ds_client.jobs[JOB_C_NAME]
print(f"Status: {ds_job_c.status}")

### 5.5 Read the output of the PDF size job

In [ ]:
ds_job_e = ds_client.jobs[JOB_E_NAME]
print(f"Status: {ds_job_e.status}")
print(f"Output files: {ds_job_e.output_paths}")

if ds_job_e.output_paths:
    print(f"\nResult:\n{ds_job_e.output_paths[0].read_text()}")

### 5.6 View the failed job status (missing dependency)

In [ ]:
ds_job_f = ds_client.jobs[JOB_F_NAME]
print(f"Status: {ds_job_f.status}")
print(f"Reason: {ds_job_f.reason}")

---

## 6. Cleanup

Remove generated test files.

In [ ]:
!rm -rf data/PDFs data/manifest.csv data/readme.md